# STEP05 — MOTH EXP047-B official TEST inference + submission

Финальный шаг для первого вида `ilPloInte3.2`.

Здесь **нет обучения и нет выбора гиперпараметров**.

Используется ровно frozen EXP047-B @ global step 70k:
- checkpoint SHA-256 проверяется;
- score = `sigmoid(presence_logit)`;
- minus prediction разворачивается обратно в genomic left→right;
- scores собираются строго в official template order:
  **сначала все `+`, затем все `-`**;
- финальный файл содержит **одно число [0,1] на строку, ровно 5 знаков после запятой**.

Намеренно **не добавляем rank-power / alpha post-processing**:
цель этого submission — внешняя оценка именно нашей зафиксированной EXP047-B линии,
а не нового post-processing эксперимента.

> **v2 correction:** `split.intervals("test")` is strand-expanded and therefore sums to 113,971,908 position-strands. For submission mapping this version reads `whitelist.test.bed.gz` directly once per genomic position and independently audits `template.test.bed.gz` as plus-then-minus.


In [1]:
from pathlib import Path
import gc
import gzip
import hashlib
import json
import math
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch_directml
from IPython.display import display

EXPECTED_PYTHON = Path(
    r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe"
).resolve()
ACTUAL_PYTHON = Path(sys.executable).resolve()
assert ACTUAL_PYTHON == EXPECTED_PYTHON, (ACTUAL_PYTHON, EXPECTED_PYTHON)

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (current, *current.parents)
    if (p / "README.md").is_file()
    and (p / "src/ml_project").is_dir()
    and (p / "data/raw/unpacked").is_dir()
)

AGENTS_PATH = next(
    (
        p for p in (
            PROJECT_ROOT / "AGENTS.md",
            PROJECT_ROOT.parent / "AGENTS.md",
        )
        if p.is_file()
    ),
    None,
)
assert AGENTS_PATH is not None

agents_sha = hashlib.sha256(AGENTS_PATH.read_bytes()).hexdigest()
EXPECTED_AGENTS_SHA = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
assert agents_sha == EXPECTED_AGENTS_SHA, (agents_sha, EXPECTED_AGENTS_SHA)

LOADER_PATH = PROJECT_ROOT / "src/ml_project/epic_data.py"
loader_sha = hashlib.sha256(LOADER_PATH.read_bytes()).hexdigest()
EXPECTED_LOADER_SHA = "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468"
assert loader_sha == EXPECTED_LOADER_SHA, (loader_sha, EXPECTED_LOADER_SHA)

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    forward_both_strands_multitask,
)
from ml_project.epic_data import SplitConfig, load_split

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def atomic_json(path, payload):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(tmp, path)

def sigmoid_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out.astype(np.float32)

print("Python:", ACTUAL_PYTHON)
print("Project:", PROJECT_ROOT)
print("GPU:", torch_directml.device_name(0))
print("AGENTS SHA:", agents_sha)
print("Loader SHA:", loader_sha)


Python: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
Project: D:\Projects\EPIC\EPIC
GPU: AMD Radeon RX 6750 GRE 12GB 
AGENTS SHA: bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3
Loader SHA: 37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468


## 1. Load frozen EXP047-B and official test split


In [2]:
ASSEMBLY = "ilPloInte3.2"
EXPECTED_TEST_BP = 56_985_954
EXPECTED_TEST_POSITION_STRAND = 113_971_908

SPLIT_CONFIG = SplitConfig(
    assembly=ASSEMBLY,
    split_version="contig_holdout_v1",
    seed=42,
    validation_contig_fraction=0.25,
)

split = load_split(PROJECT_ROOT, SPLIT_CONFIG, verify_hashes=True)
sequences, _ = load_baseline_sequences(split)

FINAL_CHECKPOINT = (
    PROJECT_ROOT
    / "artifacts/experiments"
    / "SUBMIT_ilPloInte3p2_EXP047B_RISK_CONTINUATION"
    / "run_001/branch_B/branch_B_final.pt"
)
assert FINAL_CHECKPOINT.is_file(), FINAL_CHECKPOINT

final_sha = sha256_file(FINAL_CHECKPOINT)
EXPECTED_FINAL_SHA = "975adb5184bff1ef5c760db94d69ac016e9b55084a3a0dfefab2a8e9d32cfed5"
assert final_sha == EXPECTED_FINAL_SHA, (final_sha, EXPECTED_FINAL_SHA)

checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

assert checkpoint["assembly"] == ASSEMBLY
assert checkpoint["branch"] == "B"
assert int(checkpoint["local_step"]) == 20_000
assert int(checkpoint["step"]) == 70_000
assert checkpoint["architecture"] == "rf1029_width512"

profile_config = ProfileWindowConfig(**checkpoint["profile_config"])

# IMPORTANT:
# split.intervals("test") is strand-expanded in this project, therefore its
# interval lengths sum to POSITION-STRAND count (2 x whitelist bp).
# For submission linearisation we need the strand-independent official
# whitelist exactly once, in its original file order.
WHITELIST_TEST = split.paths.inputs["whitelist_test"]
TEMPLATE_TEST = split.paths.inputs["template_test"]

test_intervals = pd.read_csv(
    WHITELIST_TEST,
    sep="\t",
    header=None,
    comment="#",
    usecols=[0, 1, 2],
    names=["contig", "start", "end"],
)
test_intervals["contig"] = test_intervals["contig"].astype(str)
test_intervals["start"] = pd.to_numeric(
    test_intervals["start"], errors="raise"
).astype(np.int64)
test_intervals["end"] = pd.to_numeric(
    test_intervals["end"], errors="raise"
).astype(np.int64)

test_intervals["length"] = (
    test_intervals["end"] - test_intervals["start"]
)
assert (test_intervals["length"] > 0).all()
assert int(test_intervals["length"].sum()) == EXPECTED_TEST_BP, (
    int(test_intervals["length"].sum()),
    EXPECTED_TEST_BP,
)

# Linear offset in the official per-strand whitelist expansion.
test_intervals["linear_start"] = np.r_[
    np.int64(0),
    np.cumsum(
        test_intervals["length"].to_numpy(np.int64)[:-1],
        dtype=np.int64,
    ),
]
test_intervals["linear_end"] = (
    test_intervals["linear_start"] + test_intervals["length"]
)

# Audit the supplied template itself: it must be exactly whitelist(+)
# followed by the same whitelist(-), matching the challenge contract.
template_frame = pd.read_csv(
    TEMPLATE_TEST,
    sep="\t",
    header=None,
    comment="#",
    dtype=str,
)
n_whitelist = len(test_intervals)
assert len(template_frame) == 2 * n_whitelist, (
    len(template_frame),
    2 * n_whitelist,
)

for half_start in (0, n_whitelist):
    half = template_frame.iloc[
        half_start:half_start + n_whitelist
    ].reset_index(drop=True)
    assert np.array_equal(
        half.iloc[:, 0].astype(str).to_numpy(),
        test_intervals["contig"].to_numpy(),
    )
    assert np.array_equal(
        pd.to_numeric(half.iloc[:, 1], errors="raise")
        .to_numpy(np.int64),
        test_intervals["start"].to_numpy(np.int64),
    )
    assert np.array_equal(
        pd.to_numeric(half.iloc[:, 2], errors="raise")
        .to_numpy(np.int64),
        test_intervals["end"].to_numpy(np.int64),
    )

strand_columns = []
for column in template_frame.columns[3:]:
    values = set(template_frame[column].dropna().astype(str).unique())
    if values and values <= {"+", "-"}:
        strand_columns.append(column)

assert len(strand_columns) == 1, (
    "Could not identify exactly one strand column in template.test.bed.gz",
    strand_columns,
)
strand_column = strand_columns[0]
assert (
    template_frame.iloc[:n_whitelist][strand_column]
    .astype(str)
    .eq("+")
    .all()
)
assert (
    template_frame.iloc[n_whitelist:][strand_column]
    .astype(str)
    .eq("-")
    .all()
)

print(
    "Official template audit: PASS |",
    f"whitelist intervals={n_whitelist:,} |",
    f"whitelist bp={int(test_intervals['length'].sum()):,}",
)

test_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="test",
    config=profile_config,
    seed=42,
)
test_tiles = test_generator.tile_index.reset_index(drop=True).copy()

required_columns = {
    "tile_id",
    "contig",
    "target_start",
    "target_end",
    "allowed_position_strand",
}
assert required_columns <= set(test_tiles.columns)

assert np.array_equal(
    test_tiles["tile_id"].to_numpy(np.int64),
    np.arange(len(test_tiles), dtype=np.int64),
)
assert int(test_tiles["allowed_position_strand"].sum()) == EXPECTED_TEST_POSITION_STRAND
assert split.positive_targets("test") is None

for contig, frame in test_tiles.groupby("contig", sort=False):
    frame = frame.sort_values("target_start")
    starts = frame["target_start"].to_numpy(np.int64)
    ends = frame["target_end"].to_numpy(np.int64)
    if len(frame) > 1:
        assert np.all(starts[1:] >= ends[:-1]), f"Overlapping target tiles: {contig}"

display(split.summary)
print("Checkpoint SHA:", final_sha)
print("Test whitelist bp:", f"{EXPECTED_TEST_BP:,}")
print("Submission lines:", f"{EXPECTED_TEST_POSITION_STRAND:,}")
print("Test tiles:", f"{len(test_tiles):,}")


Official template audit: PASS | whitelist intervals=33,717 | whitelist bp=56,985,954


ValueError: Для 'test' отсутствуют положительные targets

## 2. Build exact whitelist→linear mapping

Мы не предполагаем, что строки `tile_index` сами по себе равны submission order.

Для каждого contig сохраняем official test whitelist intervals и их cumulative offsets.
Затем prediction каждого tile раскладывается по пересечениям с whitelist.


In [ ]:
interval_map = {}

for contig, frame in test_intervals.groupby("contig", sort=False):
    frame = frame.sort_values(["start", "end"], kind="stable").copy()

    starts = frame["start"].to_numpy(np.int64)
    ends = frame["end"].to_numpy(np.int64)
    linear_starts = frame["linear_start"].to_numpy(np.int64)

    if len(frame) > 1:
        assert np.all(starts[1:] >= ends[:-1]), (
            f"Overlapping official whitelist intervals on {contig}"
        )

    interval_map[str(contig)] = {
        "start": starts,
        "end": ends,
        "linear_start": linear_starts,
    }

assert set(test_tiles["contig"].astype(str)) <= set(interval_map)

print("Contigs mapped:", len(interval_map))


## 3. TEST inference → exact raw score memmap

On-disk array shape:

```text
[2, 56,985,954]
  0 = plus
  1 = minus
```

Слой `coverage` обязан завершиться **ровно единицами** по всем 56,985,954 positions.
Это проверяет отсутствие пропусков и дублей в coordinate mapping.

Ячейка resumable по tile index.


In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts/submissions"
    / ASSEMBLY
    / "EXP047B_step70000"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_SCORE_PATH = OUTPUT_DIR / "test_presence_probability_f32.dat"
COVERAGE_PATH = OUTPUT_DIR / "_coverage_u8.dat"
INFERENCE_STATE_PATH = OUTPUT_DIR / "inference_state.json"

RAW_SHAPE = (2, EXPECTED_TEST_BP)
expected_raw_bytes = int(np.prod(RAW_SHAPE, dtype=np.int64)) * 4
expected_coverage_bytes = EXPECTED_TEST_BP

if INFERENCE_STATE_PATH.is_file():
    state = json.loads(INFERENCE_STATE_PATH.read_text(encoding="utf-8"))
    assert state["checkpoint_sha256"] == final_sha
    assert tuple(state["raw_shape"]) == RAW_SHAPE
    assert RAW_SCORE_PATH.is_file()
    assert COVERAGE_PATH.is_file()
    assert RAW_SCORE_PATH.stat().st_size == expected_raw_bytes
    assert COVERAGE_PATH.stat().st_size == expected_coverage_bytes

    completed_tiles = int(state["completed_tiles"])
    assert 0 <= completed_tiles <= len(test_tiles)

    raw_scores = np.memmap(
        RAW_SCORE_PATH,
        mode="r+",
        dtype=np.float32,
        shape=RAW_SHAPE,
    )
    coverage = np.memmap(
        COVERAGE_PATH,
        mode="r+",
        dtype=np.uint8,
        shape=(EXPECTED_TEST_BP,),
    )

    print("Resume at test tile:", completed_tiles)

else:
    if RAW_SCORE_PATH.exists() or COVERAGE_PATH.exists():
        raise RuntimeError(
            "Inference artifacts exist without state manifest. "
            "Refusing ambiguous overwrite."
        )

    raw_scores = np.memmap(
        RAW_SCORE_PATH,
        mode="w+",
        dtype=np.float32,
        shape=RAW_SHAPE,
    )
    raw_scores[:] = np.nan

    coverage = np.memmap(
        COVERAGE_PATH,
        mode="w+",
        dtype=np.uint8,
        shape=(EXPECTED_TEST_BP,),
    )
    coverage[:] = 0

    completed_tiles = 0

    state = {
        "complete": False,
        "assembly": ASSEMBLY,
        "checkpoint": str(FINAL_CHECKPOINT),
        "checkpoint_sha256": final_sha,
        "raw_shape": list(RAW_SHAPE),
        "raw_dtype": "float32",
        "score_definition": "sigmoid(EXP047-B presence logit)",
        "submission_order": "all plus positions, then all minus positions",
        "completed_tiles": 0,
    }
    atomic_json(INFERENCE_STATE_PATH, state)

if completed_tiles < len(test_tiles):
    device = torch_directml.device()

    model = create_cnn_presence_intensity(channels=512)
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    EVAL_BATCH_SIZE = 1
    started = time.perf_counter()
    initial_completed = completed_tiles

    with torch.no_grad():
        for start_idx in range(
            completed_tiles,
            len(test_tiles),
            EVAL_BATCH_SIZE,
        ):
            end_idx = min(
                start_idx + EVAL_BATCH_SIZE,
                len(test_tiles),
            )
            selected = test_tiles.iloc[start_idx:end_idx]
            batch = test_generator.make_batch(selected)

            batch_tile_ids = batch.tile_id.numpy().astype(
                np.int64,
                copy=False,
            )
            expected_ids = selected["tile_id"].to_numpy(np.int64)
            assert np.array_equal(batch_tile_ids, expected_ids)

            x_gpu = batch.x_both_strands.to(device)

            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                (
                    plus_gpu,
                    minus_reverse_gpu,
                    _,
                    _,
                ) = forward_both_strands_multitask(
                    model,
                    x_gpu,
                    batch_size=len(selected),
                    context_flank=profile_config.context_flank,
                    target_length=profile_config.target_length,
                )

            fallbacks = [
                str(item.message)
                for item in caught
                if "fall back" in str(item.message).lower()
            ]
            if fallbacks:
                raise RuntimeError(
                    "DirectML CPU fallback during test inference: "
                    + " | ".join(fallbacks)
                )

            plus_probability = sigmoid_numpy(
                plus_gpu[:, 0, :].detach().cpu().numpy()
            )
            minus_probability = sigmoid_numpy(
                minus_reverse_gpu[:, 0, :]
                .detach()
                .cpu()
                .numpy()[:, ::-1]
            )

            for local_i, (_, row) in enumerate(selected.iterrows()):
                contig = str(row["contig"])
                tile_start = int(row["target_start"])
                tile_end = int(row["target_end"])

                info = interval_map[contig]
                starts = info["start"]
                ends = info["end"]
                linear_starts = info["linear_start"]

                first = int(np.searchsorted(
                    ends,
                    tile_start,
                    side="right",
                ))
                last = int(np.searchsorted(
                    starts,
                    tile_end,
                    side="left",
                ))

                for j in range(first, last):
                    overlap_start = max(tile_start, int(starts[j]))
                    overlap_end = min(tile_end, int(ends[j]))
                    if overlap_end <= overlap_start:
                        continue

                    src_start = overlap_start - tile_start
                    src_end = overlap_end - tile_start

                    dst_start = (
                        int(linear_starts[j])
                        + overlap_start
                        - int(starts[j])
                    )
                    dst_end = dst_start + (
                        overlap_end - overlap_start
                    )

                    if np.any(coverage[dst_start:dst_end] != 0):
                        raise AssertionError(
                            "Duplicate assignment detected: "
                            f"{contig}:{overlap_start}-{overlap_end}"
                        )

                    raw_scores[
                        0,
                        dst_start:dst_end,
                    ] = plus_probability[
                        local_i,
                        src_start:src_end,
                    ]
                    raw_scores[
                        1,
                        dst_start:dst_end,
                    ] = minus_probability[
                        local_i,
                        src_start:src_end,
                    ]
                    coverage[dst_start:dst_end] = 1

            completed_tiles = end_idx

            if (
                completed_tiles % 1000 < EVAL_BATCH_SIZE
                or completed_tiles == len(test_tiles)
            ):
                raw_scores.flush()
                coverage.flush()

                elapsed = time.perf_counter() - started
                done_now = completed_tiles - initial_completed
                tiles_per_sec = done_now / max(elapsed, 1e-9)
                remaining = len(test_tiles) - completed_tiles

                state["completed_tiles"] = completed_tiles
                state["complete"] = (
                    completed_tiles == len(test_tiles)
                )
                state["elapsed_seconds_current_run"] = elapsed
                atomic_json(INFERENCE_STATE_PATH, state)

                print(
                    f"test tiles "
                    f"{completed_tiles:,}/{len(test_tiles):,} "
                    f"({100*completed_tiles/len(test_tiles):.2f}%) "
                    f"tiles/s={tiles_per_sec:.2f} "
                    f"ETA={remaining/max(tiles_per_sec,1e-9)/3600:.2f}h",
                    flush=True,
                )

            del (
                batch,
                x_gpu,
                plus_gpu,
                minus_reverse_gpu,
                plus_probability,
                minus_probability,
            )

    raw_scores.flush()
    coverage.flush()

    state["completed_tiles"] = len(test_tiles)
    state["complete"] = True
    atomic_json(INFERENCE_STATE_PATH, state)

    del model
    gc.collect()

state = json.loads(INFERENCE_STATE_PATH.read_text(encoding="utf-8"))
assert state["complete"] is True
assert int(state["completed_tiles"]) == len(test_tiles)

print("TEST INFERENCE: COMPLETE")


## 4. Exhaustive inference integrity audit


In [ ]:
raw_scores.flush()
coverage.flush()

AUDIT_CHUNK = 2_000_000

coverage_bad = 0
nonfinite = 0
below_zero = 0
above_one = 0
global_min = float("inf")
global_max = float("-inf")

for start in range(0, EXPECTED_TEST_BP, AUDIT_CHUNK):
    end = min(start + AUDIT_CHUNK, EXPECTED_TEST_BP)

    cov = np.asarray(coverage[start:end])
    coverage_bad += int(np.count_nonzero(cov != 1))

    for strand in (0, 1):
        values = np.asarray(raw_scores[strand, start:end])
        finite = np.isfinite(values)

        nonfinite += int(np.count_nonzero(~finite))
        below_zero += int(np.count_nonzero(values[finite] < 0.0))
        above_one += int(np.count_nonzero(values[finite] > 1.0))

        if finite.any():
            global_min = min(
                global_min,
                float(values[finite].min()),
            )
            global_max = max(
                global_max,
                float(values[finite].max()),
            )

assert coverage_bad == 0, coverage_bad
assert nonfinite == 0, nonfinite
assert below_zero == 0, below_zero
assert above_one == 0, above_one

print("Coverage bad:", coverage_bad)
print("Non-finite:", nonfinite)
print("Score min/max:", global_min, global_max)
print("Raw prediction values:", f"{2 * EXPECTED_TEST_BP:,}")
print("INFERENCE AUDIT: PASS")


## 5. Write official `predictions.txt.gz`

Формат:
- без header;
- без координат;
- одно число на строку;
- `+` stream целиком, затем `-` stream целиком;
- ровно пять знаков после запятой.

Quantization = **тот же `round(p × 100000) / 100000`**, что использовался
при нашей local validation.


In [ ]:
SUBMISSION_PATH = OUTPUT_DIR / "predictions.txt.gz"
SUBMISSION_MANIFEST = OUTPUT_DIR / "submission_manifest.json"

WRITE_CHUNK = 250_000

if SUBMISSION_PATH.exists() and SUBMISSION_MANIFEST.exists():
    manifest = json.loads(
        SUBMISSION_MANIFEST.read_text(encoding="utf-8")
    )
    assert manifest["checkpoint_sha256"] == final_sha
    assert int(manifest["line_count"]) == EXPECTED_TEST_POSITION_STRAND
    assert manifest["submission_sha256"] == sha256_file(SUBMISSION_PATH)
    print("Existing submission verified:", SUBMISSION_PATH)

else:
    if SUBMISSION_PATH.exists() or SUBMISSION_MANIFEST.exists():
        raise RuntimeError(
            "Partial submission artifact exists; refusing overwrite."
        )

    temp_path = SUBMISSION_PATH.with_suffix(".txt.gz.tmp")

    line_count = 0
    distinct_level_bitmap = np.zeros(100_001, dtype=bool)
    first_scores = []
    last_scores = []

    started_write = time.perf_counter()

    with gzip.open(
        temp_path,
        "wt",
        encoding="ascii",
        newline="\n",
        compresslevel=1,
    ) as handle:
        for strand in (0, 1):
            for start in range(
                0,
                EXPECTED_TEST_BP,
                WRITE_CHUNK,
            ):
                end = min(
                    start + WRITE_CHUNK,
                    EXPECTED_TEST_BP,
                )
                values = np.asarray(
                    raw_scores[strand, start:end],
                    dtype=np.float64,
                )

                bins = np.rint(
                    np.clip(values, 0.0, 1.0)
                    * 100_000
                ).astype(np.int32)

                assert bins.min() >= 0
                assert bins.max() <= 100_000

                distinct_level_bitmap[
                    np.unique(bins)
                ] = True

                quantized = (
                    bins.astype(np.float64)
                    / 100_000.0
                )

                if len(first_scores) < 10:
                    need = 10 - len(first_scores)
                    first_scores.extend(
                        quantized[:need].tolist()
                    )

                last_scores = quantized[-10:].tolist()

                np.savetxt(
                    handle,
                    quantized,
                    fmt="%.5f",
                )
                line_count += len(quantized)

                if line_count % 10_000_000 < WRITE_CHUNK:
                    print(
                        f"written {line_count:,}/"
                        f"{EXPECTED_TEST_POSITION_STRAND:,}",
                        flush=True,
                    )

    assert line_count == EXPECTED_TEST_POSITION_STRAND

    os.replace(temp_path, SUBMISSION_PATH)

    submission_sha = sha256_file(SUBMISSION_PATH)
    distinct_levels = int(distinct_level_bitmap.sum())

    manifest = {
        "assembly": ASSEMBLY,
        "model": "EXP047-B",
        "global_step": 70_000,
        "checkpoint": str(FINAL_CHECKPOINT),
        "checkpoint_sha256": final_sha,
        "score_definition": "sigmoid(presence_logit)",
        "postprocessing": "clip [0,1], round to 5 decimals only",
        "rank_power_transform": False,
        "order": "all plus whitelist positions, then all minus whitelist positions",
        "whitelist_bp_per_strand": EXPECTED_TEST_BP,
        "line_count": line_count,
        "distinct_5dp_levels": distinct_levels,
        "first_10_scores": first_scores,
        "last_10_scores": last_scores,
        "submission_path": str(SUBMISSION_PATH),
        "submission_sha256": submission_sha,
        "submission_bytes_gzip": SUBMISSION_PATH.stat().st_size,
        "write_seconds": time.perf_counter() - started_write,
    }
    atomic_json(SUBMISSION_MANIFEST, manifest)

print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 6. Independent gzip structure check

Это полный sequential read готового gzip:
- каждая строка парсится как float;
- диапазон `[0,1]`;
- ровно 5 цифр после точки;
- line count = 113,971,908.

Это может занять несколько минут, но перед первым официальным submission лучше
один раз заплатить эту цену.


In [ ]:
checked_lines = 0
parse_errors = 0
format_errors = 0
range_errors = 0

started_check = time.perf_counter()

with gzip.open(
    SUBMISSION_PATH,
    "rt",
    encoding="ascii",
    newline="",
) as handle:
    for line in handle:
        checked_lines += 1
        text = line.rstrip("\n\r")

        if (
            len(text) != 7
            or text[1] != "."
            or not text[0].isdigit()
            or not text[2:].isdigit()
        ):
            format_errors += 1
            if format_errors > 10:
                raise ValueError(
                    "Too many submission formatting errors"
                )

        try:
            value = float(text)
        except ValueError:
            parse_errors += 1
            if parse_errors > 10:
                raise
            continue

        if not (0.0 <= value <= 1.0):
            range_errors += 1

        if checked_lines % 20_000_000 == 0:
            print(
                f"validated lines: {checked_lines:,}",
                flush=True,
            )

assert checked_lines == EXPECTED_TEST_POSITION_STRAND
assert parse_errors == 0
assert format_errors == 0
assert range_errors == 0

print("Lines:", f"{checked_lines:,}")
print("Parse errors:", parse_errors)
print("Format errors:", format_errors)
print("Range errors:", range_errors)
print(
    "Validation seconds:",
    round(time.perf_counter() - started_check, 1),
)
print("SUBMISSION STRUCTURE: PASS")


## 7. Final handoff

Если предыдущая ячейка закончилась `SUBMISSION STRUCTURE: PASS`,
загружать на EPIC нужно файл:

`artifacts/submissions/ilPloInte3.2/EXP047B_step70000/predictions.txt.gz`

После загрузки сохрани leaderboard result отдельно. Не меняем этот файл задним числом.


In [ ]:
print("=" * 72)
print("READY FOR OFFICIAL EPIC SUBMISSION")
print("=" * 72)
print("Assembly:", ASSEMBLY)
print("Model: EXP047-B @70k")
print("Submission:", SUBMISSION_PATH)
print("SHA256:", sha256_file(SUBMISSION_PATH))
print("Lines:", f"{EXPECTED_TEST_POSITION_STRAND:,}")
print("Manifest:", SUBMISSION_MANIFEST)
